<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Market_regime_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# K-MEANS STOCK MARKET REGIME CLUSTERING
# NIFTY 50 | Silhouette Analysis + K-Means
# ============================================================

!pip -q install yfinance scikit-learn matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 1. DOWNLOAD NIFTY 50 DATA
# ------------------------------------------------------------

ticker = "^NSEI"

df = yf.download(
    ticker,
    start="2015-01-01",
    end="2026-09-17",
    auto_adjust=True,
    progress=False
)

# Handle MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Close", "Volume"]].dropna()

# ------------------------------------------------------------
# 2. CREATE MARKET FEATURES
# ------------------------------------------------------------

df["Return"] = df["Close"].pct_change()

# 20-day annualized volatility
df["Volatility_20D"] = (
    df["Return"]
    .rolling(20)
    .std() * np.sqrt(252)
)

# 60-day annualized volatility
df["Volatility_60D"] = (
    df["Return"]
    .rolling(60)
    .std() * np.sqrt(252)
)

# 20-day momentum
df["Momentum_20D"] = (
    df["Close"] / df["Close"].shift(20) - 1
)

# Volume change
df["Volume_Change"] = (
    df["Volume"] / df["Volume"].rolling(20).mean() - 1
)

# Remove missing values
df = df.dropna()

features = [
    "Return",
    "Volatility_20D",
    "Volatility_60D",
    "Momentum_20D",
    "Volume_Change"
]

X = df[features].copy()

# ------------------------------------------------------------
# 3. STANDARDIZE FEATURES
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("=" * 70)
print("K-MEANS STOCK MARKET REGIME CLUSTERING")
print("=" * 70)

print(f"Observations : {len(df):,}")
print(f"Features     : {len(features)}")
print(f"Features     : {', '.join(features)}")

# ------------------------------------------------------------
# 4. SILHOUETTE ANALYSIS + ELBOW METHOD
# ------------------------------------------------------------

k_values = range(2, 9)

inertias = []
silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)

evaluation = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette_Score": silhouette_scores
})

print("\nK-MEANS MODEL SELECTION")
print("-" * 70)

display(
    evaluation.style.format({
        "Inertia": "{:,.2f}",
        "Silhouette_Score": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 5. SELECT K USING SILHOUETTE SCORE
# ------------------------------------------------------------

best_k = int(
    evaluation.loc[
        evaluation["Silhouette_Score"].idxmax(),
        "K"
    ]
)

best_silhouette = evaluation[
    evaluation["K"] == best_k
]["Silhouette_Score"].iloc[0]

print("\nSELECTED NUMBER OF CLUSTERS")
print("-" * 70)
print(f"Optimal K based on Silhouette Score : {best_k}")
print(f"Silhouette Score                    : {best_silhouette:.4f}")

# ------------------------------------------------------------
# 6. ELBOW PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Inertia"],
    marker="o"
)

plt.axvline(
    best_k,
    linestyle="--",
    linewidth=2,
    label=f"Selected K = {best_k}"
)

plt.title("K-Means Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.legend()
plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 7. SILHOUETTE SCORE PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Silhouette_Score"],
    marker="o"
)

plt.axvline(
    best_k,
    linestyle="--",
    linewidth=2,
    label=f"Best K = {best_k}"
)

plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(k_values))
plt.legend()
plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 8. FINAL K-MEANS MODEL
# ------------------------------------------------------------

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)

df["Cluster"] = kmeans_final.fit_predict(
    X_scaled
)

# ------------------------------------------------------------
# 9. CLUSTER CHARACTERISTICS
# ------------------------------------------------------------

cluster_stats = df.groupby("Cluster")[features].agg([
    "mean",
    "std"
])

print("\nCLUSTER CHARACTERISTICS")
print("-" * 70)

display(
    cluster_stats.round(4)
)

# ------------------------------------------------------------
# 10. SIMPLE CLUSTER SUMMARY
# ------------------------------------------------------------

summary = df.groupby("Cluster").agg(
    Observations=("Cluster", "size"),
    Mean_Return=("Return", "mean"),
    Volatility_20D=("Volatility_20D", "mean"),
    Volatility_60D=("Volatility_60D", "mean"),
    Momentum_20D=("Momentum_20D", "mean"),
    Volume_Change=("Volume_Change", "mean")
)

summary["Frequency_%"] = (
    summary["Observations"] /
    len(df) * 100
)

print("\nREGIME SUMMARY")
print("-" * 70)

display(
    summary.style.format({
        "Mean_Return": "{:.4%}",
        "Volatility_20D": "{:.2%}",
        "Volatility_60D": "{:.2%}",
        "Momentum_20D": "{:.2%}",
        "Volume_Change": "{:.2%}",
        "Frequency_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# 11. LABEL REGIMES BASED ON CLUSTER CHARACTERISTICS
# ------------------------------------------------------------

# Use volatility + return characteristics
vol_rank = summary["Volatility_20D"].rank()

labels = {}

for cluster in summary.index:

    vol = summary.loc[
        cluster,
        "Volatility_20D"
    ]

    ret = summary.loc[
        cluster,
        "Mean_Return"
    ]

    if vol >= summary["Volatility_20D"].median():

        if ret < 0:
            regime = "High Volatility / Bearish"
        else:
            regime = "High Volatility / Bullish"

    else:

        if ret < 0:
            regime = "Low Volatility / Bearish"
        else:
            regime = "Low Volatility / Bullish"

    labels[cluster] = regime

df["Regime"] = df["Cluster"].map(labels)

print("\nCLUSTER → REGIME MAPPING")
print("-" * 70)

for cluster, regime in labels.items():
    print(f"Cluster {cluster} → {regime}")

# ------------------------------------------------------------
# 12. REGIME FREQUENCY
# ------------------------------------------------------------

regime_frequency = (
    df["Regime"]
    .value_counts()
    .to_frame("Observations")
)

regime_frequency["Frequency_%"] = (
    regime_frequency["Observations"] /
    len(df) * 100
)

print("\nREGIME FREQUENCY")
print("-" * 70)

display(
    regime_frequency.style.format({
        "Frequency_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# 13. REGIME TIMELINE
# ------------------------------------------------------------

regime_numeric = (
    df["Cluster"]
)

plt.figure(figsize=(15, 6))

plt.scatter(
    df.index,
    regime_numeric,
    c=regime_numeric,
    cmap="viridis",
    s=8
)

plt.title("NIFTY 50 — K-Means Market Regime Timeline")
plt.xlabel("Date")
plt.ylabel("Cluster")
plt.yticks(
    sorted(df["Cluster"].unique())
)

plt.grid(alpha=0.3)
plt.show()

# ------------------------------------------------------------
# 14. RETURN VS VOLATILITY CLUSTERS
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df,
    x="Volatility_20D",
    y="Return",
    hue="Regime",
    palette="viridis",
    alpha=0.6,
    s=35
)

plt.title(
    "K-Means Market Regimes: Return vs Volatility"
)

plt.xlabel("20-Day Annualized Volatility")
plt.ylabel("Daily Return")

plt.grid(alpha=0.3)
plt.show()

# ------------------------------------------------------------
# 15. REGIME PERFORMANCE
# ------------------------------------------------------------

regime_performance = df.groupby(
    "Regime"
)["Return"].agg(
    Mean_Return="mean",
    Volatility="std",
    Positive_Days=lambda x: (x > 0).mean()
)

regime_performance["Annualized_Return"] = (
    regime_performance["Mean_Return"] * 252
)

regime_performance["Annualized_Volatility"] = (
    regime_performance["Volatility"] * np.sqrt(252)
)

print("\nREGIME PERFORMANCE")
print("-" * 70)

display(
    regime_performance.style.format({
        "Mean_Return": "{:.4%}",
        "Volatility": "{:.4%}",
        "Positive_Days": "{:.2%}",
        "Annualized_Return": "{:.2%}",
        "Annualized_Volatility": "{:.2%}"
    })
)

# ------------------------------------------------------------
# 16. CURRENT MARKET REGIME
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\nCURRENT MARKET STATE")
print("-" * 70)

print(f"Date       : {df.index[-1].date()}")
print(f"Cluster    : {int(latest['Cluster'])}")
print(f"Regime     : {latest['Regime']}")
print(f"Return     : {latest['Return']:.2%}")
print(f"20D Vol    : {latest['Volatility_20D']:.2%}")
print(f"20D Momentum: {latest['Momentum_20D']:.2%}")

# ------------------------------------------------------------
# 17. SAVE RESULTS
# ------------------------------------------------------------

df.to_csv(
    "NIFTY50_KMeans_Market_Regimes.csv"
)

evaluation.to_csv(
    "KMeans_Silhouette_Evaluation.csv",
    index=False
)

summary.to_csv(
    "KMeans_Regime_Summary.csv"
)

regime_performance.to_csv(
    "KMeans_Regime_Performance.csv"
)

print("\nFiles saved:")
print("• NIFTY50_KMeans_Market_Regimes.csv")
print("• KMeans_Silhouette_Evaluation.csv")
print("• KMeans_Regime_Summary.csv")
print("• KMeans_Regime_Performance.csv")

print("\n" + "=" * 70)
print("PROJECT COMPLETED")
print("=" * 70)

K-MEANS STOCK MARKET REGIME CLUSTERING
Observations : 2,822
Features     : 5
Features     : Return, Volatility_20D, Volatility_60D, Momentum_20D, Volume_Change
